# Global Solution 1ºSemestre 2026
## Indústria Espacial: O Código que Move o Universo

 Nossa solução tem como objetivo principal: o treinamento de uma modelo de deep learning capaz de fazer a distinção entre diferentes tipos de objetos encontrados no espaço. O intuito é classificar esses objetos com suas devidas posições e comunicar os demais instrumentos ativos na órbita terrestre para que seja evitado possíveis impactos que esse lixo pode causar, além de suas graves consequências.



## Importações necessárias

In [28]:
import torch
from torch import nn
import torchvision
from torchvision import transforms, datasets
from torch.utils.data import DataLoader, random_split

In [32]:
device = torch.device("cuda"if torch.cuda.is_available() else "cpu")
print(f"Device = {device}")

Device = cpu


## Preparando dataset de imagens EuroSAT

=> Separação dos dados de treino, teste e validação
- treino = 80%
- teste = 10%
- validação = 10%

In [33]:
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

path_dados = "./dados/EuroSAT_RGB"

dataset_completo = datasets.ImageFolder(
    root=path_dados,
    transform=transform
)

print(f"Total de imagens carregadas: {len(dataset_completo)}")
print(f"Classes disponíveis: {dataset_completo.classes}")

tamanho_total = len(dataset_completo)
tam_treino = int(0.8 * tamanho_total)
tam_val = int(0.1 * tamanho_total)
tam_teste = tamanho_total - tam_treino - tam_val

dataset_treino, dataset_val, dataset_teste = random_split(
    dataset_completo, [tam_treino, tam_val, tam_teste], 
    generator = torch.Generator().manual_seed(42)
)

batch_size = 32 
loader_treino = DataLoader(dataset_treino, batch_size=batch_size, shuffle=True)
loader_val = DataLoader(dataset_val, batch_size=batch_size, shuffle=False)
loader_teste = DataLoader(dataset_teste, batch_size=batch_size, shuffle=False)

print("Dataset pronto para o treinamento!")




Total de imagens carregadas: 27000
Classes disponíveis: ['AnnualCrop', 'Forest', 'HerbaceousVegetation', 'Highway', 'Industrial', 'Pasture', 'PermanentCrop', 'Residential', 'River', 'SeaLake']
Dataset pronto para o treinamento!


## Primeira Arquitetura CNN

In [ ]:
# class CNNBaseline(nn.Module):
#     def __init__(self, num_classes=10):
#         super(CNNBaseline, self).__init__()

#         self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)
#         self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        
#         self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernle_size=3, padding=1)
        
#         self.conv3 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)

#         self.fc1 = nn.Linear(64 * 8 * 8, 128)
#         self.fc2 = nn.Linear(128, num_classes)
    
#     def forward(self, x):
#         x = self.pool(torch.nn.functional.relu(self.conv1(x)))
#         x = self.pool(torch.nn.functional.relu(self.conv2(x)))
#         x = self.pool(torch.nn.functional.relu(self.conv3(x)))

#         x = x.view(-1, 64 * 8 * 8)
#         x = torch.nn.functional.relu(self.fc1(x))
#         x = self.fc2(x)
#         return x

In [35]:
class CNNBaseline(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model1 = CNNBaseline(num_classes=10).to(device)
print(model1)


CNNBaseline(
  (features): Sequential(
    (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=4096, out_features=128, bias=True)
    (2): ReLU()
    (3): Linear(in_features=128, out_features=10, bias=True)
  )
)


# Segunda Arquitetura CNN

In [37]:
class CNNAvancada(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout(0.25),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout(0.25),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout(0.4)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 8 * 8, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )
    
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x
    
model2 = CNNAvancada(num_classes=10).to(device)
print(model2)

CNNAvancada(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (5): ReLU()
    (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (7): Dropout(p=0.25, inplace=False)
    (8): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (10): ReLU()
    (11): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (12): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (13): ReLU()
    (14): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ce

## Criando função genérica de treinamento

In [40]:
def treinar_modelo(modelo, train_loader, val_loader, criterion, optimizer, epochs=15):
    historico = {
        'train_loss': [], 'train_acc': [],
        'val_loss': [], 'val_acc': []
    }

    for epoch in range(epochs):
        modelo.train()
        running_loss = 0.0
        correct_train = 0
        total_train = 0

        for imagens, labels in train_loader:
            imagens, labels = imagens.to(device), labels.to(device)

            optimizer.zero_grad()

            outputs = modelo(imagens)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            running_loss += loss.item() * imagens.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total_train += labels.size(0)
            correct_train += (predicted == labels).sum().item()

        epoch_train_loss = running_loss / len(train_loader.dataset)
        epoch_train_acc = (correct_train / total_train) * 100

        modelo.eval()
        running_val_loss = 0.0
        correct_val = 0
        total_val = 0

        with torch.no_grad():

            for imagens, labels in val_loader:
                imagens, labels = imagens.to(device), labels.to(device)

                outputs = modelo(imagens)
                loss = criterion(outputs, labels)

                running_val_loss += loss.item() * imagens.size(0)
                _, predicted = torch.max(outputs.data, 1)
                total_val += labels.size(0)
                correct_val += (predicted == labels).sum().item()

        epoch_val_loss = running_val_loss / len(val_loader.dataset)
        epoch_val_acc = (correct_val / total_val) * 100

        historico['train_loss'].append(epoch_train_loss)
        historico['train_acc'].append(epoch_train_acc)
        historico['val_loss'].append(epoch_val_loss)
        historico['val_acc'].append(epoch_val_acc)

        print(f"Época [{epoch+1}/{epochs}] | "
              f"Treino Loss: {epoch_train_loss:.4f} Acc: {epoch_train_acc:.2f}% | "
              f"Val Loss: {epoch_val_loss:.4f} Acc: {epoch_val_acc:.2f}%")
    
    return historico

In [41]:
criterion = nn.CrossEntropyLoss()
num_epochs = 15

print("Treinando modelo 1")
otimizador1 = torch.optim.Adam(model1.parameters(), lr=0.001)
historico1 = treinar_modelo(model1, loader_treino, loader_val, criterion, otimizador1, epochs=num_epochs)

print("Treinando modelo 2")
otimizador2 = torch.optim.Adam(model2.parameters(), lr=0.0005, weight_decay=1e-4)
historico2 = treinar_modelo(model2, loader_treino, loader_val, criterion, otimizador2, epochs=num_epochs)

Treinando modelo 1
Época [1/15] | Treino Loss: 0.6179 Acc: 77.70% | Val Loss: 0.5465 Acc: 79.63%
Época [2/15] | Treino Loss: 0.4898 Acc: 82.42% | Val Loss: 0.4670 Acc: 83.63%
Época [3/15] | Treino Loss: 0.4094 Acc: 85.47% | Val Loss: 0.4296 Acc: 84.96%
Época [4/15] | Treino Loss: 0.3347 Acc: 88.10% | Val Loss: 0.4040 Acc: 85.52%
Época [5/15] | Treino Loss: 0.2668 Acc: 90.64% | Val Loss: 0.3793 Acc: 87.33%
Época [6/15] | Treino Loss: 0.2246 Acc: 92.13% | Val Loss: 0.3769 Acc: 87.52%
Época [7/15] | Treino Loss: 0.1923 Acc: 93.25% | Val Loss: 0.4045 Acc: 87.00%
Época [8/15] | Treino Loss: 0.1615 Acc: 94.23% | Val Loss: 0.3758 Acc: 88.74%
Época [9/15] | Treino Loss: 0.1155 Acc: 96.06% | Val Loss: 0.4543 Acc: 87.48%
Época [10/15] | Treino Loss: 0.1112 Acc: 96.12% | Val Loss: 0.5139 Acc: 87.00%
Época [11/15] | Treino Loss: 0.0949 Acc: 96.73% | Val Loss: 0.4567 Acc: 87.41%
Época [12/15] | Treino Loss: 0.0799 Acc: 97.33% | Val Loss: 0.4474 Acc: 89.00%
Época [13/15] | Treino Loss: 0.0678 Acc: 9

KeyboardInterrupt: 